# Model Application & Failure Analysis
## Customized solution with mathematical failure diagnosis.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
warnings.filterwarnings('ignore')
np.random.seed(42)
PALETTE = ["#4E79A7","#F28E2B","#E15759","#76B7B2","#59A14F"]

In [ ]:
from datasets import load_dataset
ds = load_dataset('nkazi/MohlerASAG')
records = []
for split in ds.keys():
    for row in ds[split]:
        if not row.get('id'): continue
        q_id = '.'.join(row['id'].split('.')[:2])
        records.append({'question_id': q_id, 'answer': str(row['student_answer']).replace('<br>','').strip(), 'score': float(row.get('score_avg', 0))})
df = pd.DataFrame(records)
df['word_count'] = df['answer'].str.split().str.len()
df['label'] = (df['score'] >= 3).astype(int)

ref_answers = df.groupby('question_id')['answer'].first().to_dict()
def get_tfidf_sim(row):
    ref = ref_answers.get(row['question_id'], '')
    if not ref: return np.nan
    try:
        vec = TfidfVectorizer().fit_transform([ref, row['answer']])
        return float(cosine_similarity(vec[0], vec[1])[0,0])
    except: return np.nan
df['tfidf_sim'] = df.apply(get_tfidf_sim, axis=1)

df['log_wc'] = np.log1p(df['word_count'])
lr = LinearRegression().fit(df[['log_wc']], df['tfidf_sim'])
df['feat_lds'] = df['tfidf_sim'] - lr.predict(df[['log_wc']])

q_wrong_mean = df[df['label'] == 0].groupby('question_id')['tfidf_sim'].mean()
df = df.merge(q_wrong_mean.rename('wrong_mean').reset_index(), on='question_id', how='left')
df['wrong_mean'] = df['wrong_mean'].fillna(df['tfidf_sim'].mean() * 0.3)
df['feat_margin'] = df['tfidf_sim'] - df['wrong_mean']

FEATURES = ['tfidf_sim', 'feat_margin', 'feat_lds']
X = df[FEATURES].fillna(0).values
y = df['label'].values
print(f"Data loaded: {len(df)} samples")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_baseline = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression())])
pipe_custom = Pipeline([
    ('qt', QuantileTransformer(output_distribution='normal', random_state=42)),
    ('clf', LogisticRegression(C=0.316, class_weight='balanced', random_state=42, max_iter=1000))
])

probs_base = cross_val_predict(pipe_baseline, X, y, cv=cv, method='predict_proba')[:,1]
probs_custom = cross_val_predict(pipe_custom, X, y, cv=cv, method='predict_proba')[:,1]

thresholds = np.arange(0.2, 0.8, 0.01)
f1_scores = [f1_score(y, (probs_custom >= t).astype(int), zero_division=0) for t in thresholds]
opt_thr = thresholds[np.argmax(f1_scores)]
y_pred = (probs_custom >= opt_thr).astype(int)

print('MODEL COMPARISON')
print(f"Baseline AUC: {roc_auc_score(y, probs_base):.4f}")
print(f"Custom AUC:   {roc_auc_score(y, probs_custom):.4f}")
print(f"Custom F1:    {f1_score(y, y_pred):.4f} (threshold={opt_thr:.2f})")
print()
print(classification_report(y, y_pred, target_names=['Fail', 'Pass']))

In [ ]:
df['pred_prob'] = probs_custom
df['pred_label'] = y_pred
df['error_type'] = 'TN'
df.loc[(df['pred_label']==1)&(df['label']==1), 'error_type'] = 'TP'
df.loc[(df['pred_label']==1)&(df['label']==0), 'error_type'] = 'FP'
df.loc[(df['pred_label']==0)&(df['label']==1), 'error_type'] = 'FN'

fp_df = df[df['error_type'] == 'FP']
fn_df = df[df['error_type'] == 'FN']

print('FAILURE ANALYSIS')
print(f"Total errors: {len(fp_df)} FP + {len(fn_df)} FN = {len(fp_df)+len(fn_df)}")
print(f"Error rate: {(len(fp_df)+len(fn_df))/len(df)*100:.1f}%")

In [ ]:
print('\n--- FP Type 1: Surface Match ---')
print('Condition: High TF-IDF similarity but label=0')
print('Mathematical cause: TF-IDF = lexical overlap, not semantic correctness')
if len(fp_df) > 0:
    overlap_fp = len(fp_df[(fp_df['tfidf_sim'] >= 0.2) & (fp_df['tfidf_sim'] <= 0.5)])
    print(f'In overlap zone (0.2-0.5): {overlap_fp} cases')
print('Fix: Similarity Margin feature')

In [ ]:
print('\n--- FN Type 1: Correct but Short ---')
print('Condition: Low TF-IDF similarity but label=1')
print('Mathematical cause: TF-IDF cosine penalizes short answers')
if len(fn_df) > 0:
    short_fn = len(fn_df[fn_df['word_count'] < df['word_count'].quantile(0.25)])
    print(f'Short answers (Q1): {short_fn} cases')
print('Fix: Length-Decoupled Similarity feature')

In [ ]:
ceiling = max([f1_score(y, (df['tfidf_sim'] >= t).astype(int), zero_division=0) for t in np.arange(0.05, 0.95, 0.01)])
print(f'\nTF-IDF F1 ceiling: {ceiling:.4f}')
print(f'Custom pipeline F1: {f1_score(y, y_pred):.4f}')
print(f'Gap: {ceiling - f1_score(y, y_pred):.4f} (requires semantic embeddings)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

ax = axes[0, 0]
fpr_b, tpr_b, _ = roc_curve(y, probs_base)
fpr_c, tpr_c, _ = roc_curve(y, probs_custom)
ax.plot(fpr_b, tpr_b, color=PALETTE[2], lw=2, label=f'Baseline ({roc_auc_score(y, probs_base):.3f})')
ax.plot(fpr_c, tpr_c, color=PALETTE[0], lw=2, label=f'Custom ({roc_auc_score(y, probs_custom):.3f})')
ax.plot([0,1], [0,1], 'gray', linestyle=':', alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Comparison'); ax.legend()

ax = axes[0, 1]
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
            xticklabels=['Pred Fail', 'Pred Pass'], 
            yticklabels=['True Fail', 'True Pass'])
ax.set_title('Confusion Matrix')

ax = axes[1, 0]
error_counts = df['error_type'].value_counts()
colors_err = {'TP': PALETTE[0], 'TN': PALETTE[3], 'FP': PALETTE[2], 'FN': PALETTE[1]}
ax.bar(error_counts.index, error_counts.values, 
       color=[colors_err.get(e, 'gray') for e in error_counts.index])
ax.set_ylabel('Count'); ax.set_title('Error Type Distribution')

ax = axes[1, 1]
ax.scatter(fp_df['tfidf_sim'], fp_df['word_count'], 
           color=PALETTE[2], alpha=0.6, s=50, label=f'FP (n={len(fp_df)})')
ax.scatter(fn_df['tfidf_sim'], fn_df['word_count'], 
           color=PALETTE[1], alpha=0.6, s=50, label=f'FN (n={len(fn_df)})')
ax.set_xlabel('TF-IDF Similarity'); ax.set_ylabel('Word Count')
ax.set_title('Failure Map'); ax.legend()

plt.tight_layout()
plt.show()